In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch

In [2]:
df = pd.read_csv("/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_train.csv")
df.head(2)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
X = df.drop(columns=['label']).values
y = df['label'].values

X = X/255.0

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

# Custom Dataset and loader

In [7]:
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch
import torch.optim as optim

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [8]:
class CustomDataset(Dataset):
  def __init__(self,X,y):
    self.features = torch.tensor(X,dtype=torch.float32)
    self.labels = torch.tensor(y,dtype=torch.long)
      
  def __len__(self):
    return len(self.features)
      
  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [9]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

train_dataloader = DataLoader(train_dataset,shuffle=True,pin_memory=True,batch_size=64)
test_dataloader = DataLoader(test_dataset,pin_memory=True,batch_size=20)

# Model building

In [12]:
class MyNN(nn.Module):
  def __init__(self,input_features):
    super().__init__()

    self.linear1 = nn.Linear(input_features,128)
    self.linear2 = nn.Linear(128,64)
    self.linear3 = nn.Linear(64,10)
    self.gelu = nn.GELU()
    self.batch1 = nn.BatchNorm1d(128)
    self.batch2 = nn.BatchNorm1d(64)
    self.drop1 = nn.Dropout(p=0.3)

  def forward(self,x):
    y_pred = self.linear1(x)
    y_pred = self.batch1(y_pred)
    y_pred = self.gelu(y_pred)
    y_pred = self.drop1(y_pred)
      
    y_pred = self.linear2(y_pred)
    y_pred = self.batch2(y_pred)
    y_pred = self.gelu(y_pred)
    y_pred = self.drop1(y_pred)
      
    y_pred = self.linear3(y_pred)

    return y_pred

In [22]:
model = MyNN(X_test.shape[1])
model = model.to(device)

learning_rate = 0.0001
epochs = 50
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=learning_rate,weight_decay=1e-4)

# Training

In [23]:
for epoch in range(epochs):
  total_loss=0
  for batch_features,batch_labels in train_dataloader:
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
      
    y_pred = model(batch_features)
    loss = loss_function(y_pred,batch_labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  print(f"epoch: {epoch+1} loss: {total_loss/len(train_dataloader)}")

epoch: 1 loss: 1.090552111784617
epoch: 2 loss: 0.6338721435864766
epoch: 3 loss: 0.5224014860788981
epoch: 4 loss: 0.471323102970918
epoch: 5 loss: 0.43710817325115203
epoch: 6 loss: 0.4174949720899264
epoch: 7 loss: 0.40155949304501215
epoch: 8 loss: 0.3901284877260526
epoch: 9 loss: 0.37305630022287367
epoch: 10 loss: 0.3639539056221644
epoch: 11 loss: 0.3560892977118492
epoch: 12 loss: 0.3479566084742546
epoch: 13 loss: 0.341261511405309
epoch: 14 loss: 0.3354700089097023
epoch: 15 loss: 0.32694909554719925
epoch: 16 loss: 0.321917688190937
epoch: 17 loss: 0.31882246935367586
epoch: 18 loss: 0.3114842313329379
epoch: 19 loss: 0.3072966287434101
epoch: 20 loss: 0.3024034363925457
epoch: 21 loss: 0.29855313278237977
epoch: 22 loss: 0.29644682683547335
epoch: 23 loss: 0.2899826470812162
epoch: 24 loss: 0.28546515957514446
epoch: 25 loss: 0.28244650209943456
epoch: 26 loss: 0.2789961003363132
epoch: 27 loss: 0.27603236604730286
epoch: 28 loss: 0.2755586442450682
epoch: 29 loss: 0.27113

# Evaluation

In [24]:
model.eval()

MyNN(
  (linear1): Linear(in_features=784, out_features=128, bias=True)
  (linear2): Linear(in_features=128, out_features=64, bias=True)
  (linear3): Linear(in_features=64, out_features=10, bias=True)
  (gelu): GELU(approximate='none')
  (batch1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (batch2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop1): Dropout(p=0.3, inplace=False)
)

In [25]:
total =0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in test_dataloader:
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
    y_pred = model(batch_features)
    _,predicted = torch.max(y_pred,1)

    total += batch_features.shape[0]
    correct += (predicted==batch_labels).sum().item()

print(f"validation accuracy {correct/total}")

validation accuracy 0.8910833333333333


## ovrefit check

In [26]:
total =0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in train_dataloader:
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
    y_pred = model(batch_features)
    _,predicted = torch.max(y_pred,1)

    total += batch_features.shape[0]
    correct += (predicted==batch_labels).sum().item()

print(f"train accuracy {correct/total}")

train accuracy 0.9442708333333333


# no diff file test

In [27]:
test = pd.read_csv("/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_test.csv")
test.head(2)

test_X = test.drop(columns=['label']).values
test_y = test['label'].values

actual_test_dataset = CustomDataset(test_X,test_y)
actual_test_dataloader = DataLoader(actual_test_dataset,shuffle=False,pin_memory=True
                                   ,batch_size=64
                                   )

total =0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in actual_test_dataloader:
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
    y_pred = model(batch_features)
    _,predicted = torch.max(y_pred,1)

    total += batch_features.shape[0]
    correct += (predicted==batch_labels).sum().item()

print(f"test accuracy {correct/total}")

test accuracy 0.8277
